In [ ]:
import numpy as np
import pandas as pd
import kagglehub
import os
path=kagglehub.dataset_download('clmentbisaillon/fake-and-real-news-dataset')
print(os.listdir(path))
fake_df=pd.read_csv(os.path.join(path,"Fake.csv"))
true_df=pd.read_csv(os.path.join(path,"True.csv"))

In [ ]:
fake_df.info()
true_df.info()
print(fake_df.isna().sum())
print(true_df.isna().sum())

In [ ]:
#concat both into 1 df
fake_df['label'] = 1
true_df['label'] = 0
df=pd.concat([true_df,fake_df],ignore_index=True)
print(df.head())

print(df.info())
df = df.drop(columns=['date', 'subject'])
df['text'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
df = df.drop(columns=['title'])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.countplot(x=df["label"])
plt.title("Fake vs Real News Distribution")
plt.show()

In [ ]:
#preprocess
from nltk.tokenize import word_tokenize
from sklearn.pipeline import Pipeline
from nltk.corpus import stopwords
from sklearn.compose import ColumnTransformer
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.base import BaseEstimator, TransformerMixin
import nltk

nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')
y=df["label"]
x=df.drop(columns=["label"])
class TextPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, use_lemmatizer=True):
        self.use_lemmatizer = use_lemmatizer
        self.lemmatizer = WordNetLemmatizer()
        self.stopwords_set = set(stopwords.words('english'))
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            X = X.iloc[:, 0]  # assume single column
        processed_text = []
        for text in X:
            if not isinstance(text, str):
                text = str(text) if text is not None else ''
            tokens = word_tokenize(text.lower())
            tokens = [word for word in tokens if word not in self.stopwords_set]
            tokens = [self.lemmatizer.lemmatize(word) for word in tokens]
            processed_text.append(" ".join(tokens))
        return processed_text
vectorize=TfidfVectorizer()
preprocessor=ColumnTransformer(
    transformers=[
        ("word", Pipeline([("transform",TextPreprocessor()), ("vectorize", vectorize)]), x.columns)
    ]
)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
x_train,x_test,y_train,y_test=train_test_split(x,y,train_size=0.8,shuffle=True)
model=LogisticRegression()
my_pipeline=Pipeline(
    steps=[
        ("preprocess",preprocessor),
        ("model",model)
    ]
)
my_pipeline.fit(x_train,y_train)

predictions=my_pipeline.predict(x_test)
print(accuracy_score(y_test,predictions))